# Convective heat transfer from an object

In the lecture, we performed an energy balance to determine the cooling of an object.  The final expression for the change of temperature $T(t)$ of the object over the time interval from $t$ to $t+\Delta t$ is:
\begin{align*}
T(t+\Delta t)
- T(t)
&=
- \frac{\dot{Q}(t)}{M C_p} \Delta t
\end{align*}
where $\dot{Q}(t)$ is the rate of heat loss from the object, $M$ is the mass of the object, and $C_p$ is is its specific heat capacity.  We assume that the heat loss is due to convection and depends on the temperature of the object as:
\begin{align*}
\dot{Q}(t) = hS [T(t) - T_0]
\end{align*}
where $h$ is the heat transfer coefficient, $S$ is the surface area for heat transfer, and $T_0$ is the temperature of the surroundings.

This equation can be solved analytically by separation of variables.  Below, we plot the variation of temperature with time for the example given in Q2 of [tutorial 4](https://colab.research.google.com/github/mjksill/CP213-online/blob/master/tutorials/tutorialS1_4.ipynb).

In [ ]:
h = 2       # heat transfer coefficient W m^-2 K^-1
S = 50e-4   # area for heat transfer / m^2
V = 250e-3   # L
rho = 1.0    # kg /L
M = rho * V  # mass of object / kg
Cp = 4200    # J kg^{-1} K^{-1}

tau = M * Cp /(h*S)  # heating/cooling time scale / s
print(f'tau = {tau} s')


Tinit = 60 # initial temperature of object / C
T0 = 20    #  temperature of the environment / C




import numpy as np
import matplotlib.pyplot as plt


t_list = np.linspace(0, 5*tau)
T_list = [T0 + (Tinit-T0)*np.exp(-t/tau) for t in t_list]

plt.plot(t_list, T_list)
plt.xlabel('time / s')
plt.ylabel('temperature / C')
plt.xlim([0, 5*tau])
plt.ylim([20, 60])

plt.vlines(x=tau, ymin=T0, ymax=Tinit, color='red', ls='dotted')

plt.show()





In the derivation, we defined a variable $\tau=MC_p/(hS)$ in order to simplify the algebra.  Interestingly, $\tau$ is a time scale that appears naturally from the properties of the system.  Its value is shown in the plot above as the vertical dotted line.  As we can see, it is a time scale over which "typical" changes occur.  This is how the systems "measures" things.  This basic idea will be discussed in more detail when you cover dimensional analysis in heat transfer.



## Euler's method

As mentioned in the lecture, we can approximately solve the differential equation derived from the energy balance by "stepping" through time.  This is known as the Euler's method.  Let's rewrite the balance as:
\begin{align*}
T(t+\Delta t)
&=
T(r) - \frac{\dot{Q}(t)}{M C_p} \Delta t
\\
&=
T(t) - \frac{h S[T(t)-T_0]}{M C_p} \Delta t
\\
&= T(t) - \left[\frac{T(t)-T_0}{\tau}\right] \Delta t
\end{align*}
If we know the temperature at time $t$ (i.e. if we know the value of $T(t)$), then we can use this equation to determine the temperature at time $t+\Delta t$ (i.e. the value of $T(t+\Delta t)$).

For our problem, this process starts at time $t=0$ with the given initial condition $T(0)=60^\circ\,{\rm C}$.  Once we make the first time step, then we just shift everything over by $\Delta t$ and repeating the process again and again.  The Python code below implements this method.



In [ ]:

NSTEPS = 1000  # number of time steps
dt = 0.01*tau  # time step / s

Tapprox_list = [Tinit]   # list to hold the value of temperature at each time step
tapprox_list = [0.0]

told = 0.0
Told = Tinit

for i in range(NSTEPS):
  Cpnew = Cp+0.001*Told
  Tnew = Told - (Told - T0) * dt * h*S/(M*Cpnew)
  tnew = told + dt
#  print(tnew, Tnew)
  Tapprox_list.append(Tnew)
  tapprox_list.append(tnew)
  told = tnew
  Told = Tnew


plt.plot(tapprox_list, Tapprox_list, ls='none', marker='o', color='orange',
         label="Euler's method")
plt.plot(t_list, T_list, label='exact', color='blue')
plt.legend()
plt.xlabel('time / s')
plt.ylabel('temperature / C')
plt.xlim([0, 5*tau])
plt.ylim([20, 60])
plt.show()

#print(Tapprox_list)




The approximate solution given by Euler's method is shown as the orange circles, while the exact analytical solution is shown by the blue curve.  The method appears to agree well with the exact solution.

Note that Euler's method is approximate, however, it becomes better the smaller the value of $\Delta t$.  In order to obtain an accurate solution, $\Delta t$ must be "sufficiently small".  However, if we choose too small a value of $\Delta t$, we will need to perform an extremely large amount of iterations to reach a reasonable time.  So, we need to be careful in our choice of $\Delta t$.

How small should $\Delta t$ be in order for the solution to be reasonable accurate?  As mentioned previously, the parameter $\tau$ gives a characteristic time scale by which the system "measures" itself against.  So, when we say that a time is short, we mean that it is smaller than $\tau$.  In this case, this means that in order to get an accurate solution to the equation, we need to choose $\Delta t\ll \tau$.  In this case, we choose $\Delta = 0.01\tau$.  

As $\Delta t$ becomes larger, the solution will increasingly deviate from the exact solution.  See what happens when you increase the size of the time step $\Delta t$ in the code above.